In [1]:
# Create Bloomberg request sheets with at most 250 securities per sheet.
# Keep Bloomberg sector and industry in the first security sheet; preserve the macro request.

In [2]:
# Locate shared project configuration and reusable helpers.
import sys as _sys
from pathlib import Path as _Path

_UTILS_ROOT = next(
    path
    for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
    if (path / "notebook_utils.py").is_file()
)
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from project_config import ANNUAL_FIELDS, MARKET_FIELDS, REFERENCE_FIELDS, MACRO_FIELDS
from pathlib import Path
import re

import pandas as pd
import xlsxwriter

# Locate the project root even when the kernel starts in notebooks/.
ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "data").is_dir()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Run this notebook from the project root or a folder inside it."
    )

INPUT = ROOT / "data/intermediate/bloomberg_security_list.csv"
SECURITY_DIR = ROOT / "data/intermediate/security"
MACRO_DIR = ROOT / "data/intermediate/macro"
BATCH_SIZE = 250
# Keep False to protect workbooks already populated in Bloomberg.
OVERWRITE = True

In [3]:
# Load and validate the complete security list
#
# Every input record is included exactly once. Invalid identifiers, duplicate company
# IDs, or invalid date windows stop generation instead of silently dropping records. The
# final part can contain fewer than 250 securities.


In [4]:
# Validate the complete security list before splitting it into request sheets.
companies = pd.read_csv(INPUT, dtype="string")
required = {
    "company_id",
    "company_type",
    "name",
    "bloomberg_identifier",
    "start_date",
    "end_date",
}
missing = required - set(companies.columns)
if missing:
    raise ValueError(f"Security list is missing columns: {sorted(missing)}")
if companies.empty:
    raise ValueError("The security list is empty.")
if not isinstance(BATCH_SIZE, int) or isinstance(BATCH_SIZE, bool) or BATCH_SIZE <= 0:
    raise ValueError("BATCH_SIZE must be a positive integer.")

for column in ["company_id", "bloomberg_identifier"]:
    companies[column] = companies[column].str.strip()
    invalid = companies[column].fillna("").eq("")
    if column == "bloomberg_identifier":
        invalid |= companies[column].str.upper().eq("NOT_FOUND").fillna(False)
    if invalid.any():
        raise ValueError(
            f"{column} is missing or invalid at CSV rows {(companies.index[invalid] + 2).tolist()}."
        )
if companies["company_id"].duplicated().any():
    raise ValueError(
        "company_id must be unique so every input record maps to one worksheet."
    )

date_windows = companies[["start_date", "end_date"]].apply(
    pd.to_datetime, errors="coerce"
)
invalid_dates = date_windows.isna().any(axis=1) | (
    date_windows["start_date"] > date_windows["end_date"]
)
if invalid_dates.any():
    raise ValueError(
        f"Invalid date windows at CSV rows {(companies.index[invalid_dates] + 2).tolist()}."
    )
for column in ["start_date", "end_date"]:
    companies[column] = date_windows[column].dt.strftime("%Y-%m-%d")

# One shared macro history covering every company's five-year collection window.
# Override these two values here if a different macro range is needed.
MACRO_START_DATE = date_windows["start_date"].min().strftime("%Y-%m-%d")
MACRO_END_DATE = date_windows["end_date"].max().strftime("%Y-%m-%d")

batches = [
    companies.iloc[start : start + BATCH_SIZE].copy()
    for start in range(0, len(companies), BATCH_SIZE)
]
security_paths = [
    SECURITY_DIR / f"bloomberg_addin_workbook_part_{part:03d}.xlsx"
    for part in range(1, len(batches) + 1)
]
macro_path = MACRO_DIR / "bloomberg_macro_addin_workbook.xlsx"
output_paths = [*security_paths, macro_path]
existing = [path for path in output_paths if path.exists()]
if existing and not OVERWRITE:
    raise FileExistsError(
        "Workbooks already exist. Set OVERWRITE = True only if they can be replaced:\n"
        + "\n".join(str(path) for path in existing)
    )

In [5]:
# Create unique worksheet names that obey Excel?s naming limits.
def safe_sheet_name(company_id, name, used_names):
    name = "" if pd.isna(name) else str(name)
    raw = re.sub(r"[\\/*?:\[\]]", "", f"{company_id}_{name}")[:31].strip("'") or str(
        company_id
    )
    candidate = raw
    counter = 2
    while candidate.casefold() in used_names:
        suffix = f"_{counter}"
        candidate = raw[: 31 - len(suffix)] + suffix
        counter += 1
    used_names.add(candidate.casefold())
    return candidate


def bdh_formula(security, field, start_date, end_date, periodicity):
    arguments = [
        security,
        field,
        start_date,
        end_date,
        f"Per={periodicity}",
        "Days=T",
        "Dir=V",
    ]
    return (
        "=BDH("
        + ",".join('"' + str(value).replace('"', '""') + '"' for value in arguments)
        + ")"
    )


def write_field_block(
    worksheet,
    start_col,
    alias,
    field,
    security,
    start_date,
    end_date,
    periodicity,
    width=16,
):
    worksheet.write_string(0, start_col, f"{alias}_Date")
    worksheet.write_string(0, start_col + 1, f"{alias}_Value")
    worksheet.write_formula(
        1, start_col, bdh_formula(security, field, start_date, end_date, periodicity)
    )
    worksheet.set_column(start_col, start_col + 1, width)


def write_company_workbook(batch, output):
    with xlsxwriter.Workbook(output) as workbook:
        index = workbook.add_worksheet("Security_List")
        index.write_row(0, 0, [*batch.columns, *REFERENCE_FIELDS])
        security_column = batch.columns.get_loc("bloomberg_identifier")
        for row_number, values in enumerate(
            batch.itertuples(index=False, name=None), start=1
        ):
            for column, value in enumerate(values):
                index.write_string(
                    row_number, column, "" if pd.isna(value) else str(value)
                )
            security_cell = xlsxwriter.utility.xl_rowcol_to_cell(
                row_number, security_column, col_abs=True
            )
            for column, field in enumerate(
                REFERENCE_FIELDS.values(), start=len(batch.columns)
            ):
                index.write_formula(
                    row_number, column, f'=BDP({security_cell},"{field}")', None, ""
                )
        index.set_column(0, len(batch.columns) - 1, 18)
        index.set_column(
            len(batch.columns), len(batch.columns) + len(REFERENCE_FIELDS) - 1, 32
        )

        used_names = {"security_list"}
        for company in batch.itertuples(index=False):
            sheet = workbook.add_worksheet(
                safe_sheet_name(company.company_id, company.name, used_names)
            )
            for i, (alias, field) in enumerate(ANNUAL_FIELDS.items()):
                write_field_block(
                    sheet,
                    i * 2,
                    alias,
                    field,
                    company.bloomberg_identifier,
                    company.start_date,
                    company.end_date,
                    "Y",
                )
            market_offset = len(ANNUAL_FIELDS) * 2
            for i, (alias, field) in enumerate(MARKET_FIELDS.items()):
                write_field_block(
                    sheet,
                    market_offset + i * 2,
                    alias,
                    field,
                    company.bloomberg_identifier,
                    company.start_date,
                    company.end_date,
                    "M",
                )


def write_macro_workbook(output):
    with xlsxwriter.Workbook(output) as workbook:
        sheet = workbook.add_worksheet("Macro_Data")
        for i, (alias, config) in enumerate(MACRO_FIELDS.items()):
            write_field_block(
                sheet,
                i * 2,
                alias,
                config["field"],
                config["security"],
                MACRO_START_DATE,
                MACRO_END_DATE,
                config["periodicity"],
                width=20,
            )

In [6]:
# Repeat the overwrite check here so rerunning this cell also protects saved data.
existing = [path for path in output_paths if path.exists()]
if existing and not OVERWRITE:
    raise FileExistsError(
        "Output workbooks already exist; set OVERWRITE = True to replace them."
    )

SECURITY_DIR.mkdir(parents=True, exist_ok=True)
MACRO_DIR.mkdir(parents=True, exist_ok=True)

summary = []
for part, (batch, output) in enumerate(zip(batches, security_paths), start=1):
    write_company_workbook(batch, output)
    summary.append(
        {
            "part": part,
            "securities": len(batch),
            "first_company_id": batch.iloc[0]["company_id"],
            "last_company_id": batch.iloc[-1]["company_id"],
            "workbook": output.name,
        }
    )

write_macro_workbook(macro_path)

summary = pd.DataFrame(summary)
display(summary)

,part,securities,first_company_id,last_company_id,workbook
0,1,250,SA_222,SA_249,bloomberg_addin_workbook_part_001.xlsx
1,2,250,SA_293,NB_230,bloomberg_addin_workbook_part_002.xlsx
2,3,250,NB_231,NB_480,bloomberg_addin_workbook_part_003.xlsx
3,4,250,NB_481,NB_730,bloomberg_addin_workbook_part_004.xlsx
4,5,250,NB_731,NB_980,bloomberg_addin_workbook_part_005.xlsx
5,6,250,NB_981,NB_1230,bloomberg_addin_workbook_part_006.xlsx
6,7,250,NB_1231,NB_1480,bloomberg_addin_workbook_part_007.xlsx
7,8,250,NB_1481,NB_1730,bloomberg_addin_workbook_part_008.xlsx
8,9,250,NB_1731,NB_1980,bloomberg_addin_workbook_part_009.xlsx
9,10,250,NB_1981,NB_2230,bloomberg_addin_workbook_part_010.xlsx


In [7]:
# Check that batching includes every security once, in the original order.
combined = pd.concat(batches, ignore_index=True)

In [8]:
# Retrieve the Bloomberg data
#
# 1. Open each company part and the macro workbook in Excel with the Bloomberg Add-In
# active.
# 2. Refresh the BDP sector/industry formulas on Security_List and the BDH formulas
# on each company sheet; allow each history to fill its Date/Value columns.
# 3. Save the populated workbooks. Preserve the worksheet names and paired column
# headers for later cleaning.